# BirdCLEF+ 2025 - Pretrained models, A3: MLP head on cached Bird-MAE-Base embeddings

This notebook takes one config, the best of the probe-retry sweep, runs it longer and saves it for inference. It uses the 768-d mean-pooled embeddings from `sc4000-birdclef-m3-a3-birdmae-embed`, standardised with the train mean/std, and an MLP with 512 hidden units, ReLU, dropout 0.2, AdamW, one-cycle max_lr 3e-3, batch 1024 and 30 epochs.

We train it twice with the same seed: (a) with the default final-layer bias, and (b) with the final-layer bias set to the per-class training log prior. The split, labels, window weights and validation metrics are the same as in the sweep. For each run the best epoch is kept by first-5s AUC, and the run with the higher first-5s AUC is saved (default on a tie).

CPU only, no submission. Output: `birdmae_mlp_head.npz`, `species.json`, `head_cfg.json`.


In [ ]:
import ast, glob, json, os, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

T0 = time.time()
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # local test only, never set on Kaggle
OUT = Path(os.environ.get("M3_OUT_DIR", "/kaggle/working"))
DEVICE = "cpu"       # CPU-only notebook, by design
print("device:", DEVICE, "| torch.cuda.is_available():", torch.cuda.is_available(),
      "| torch threads:", torch.get_num_threads(), "| os.cpu_count():", os.cpu_count())

# Same values as the A3 embed notebook's CFG (split, labels, training recipe)
CFG = dict(val_frac=0.2, seed=42, secondary_weight=0.5, batch_size=1024, weight_decay=1e-2)
HEAD = dict(kind="mlp", hidden=512, dropout=0.2, max_lr=3e-3, epochs=2 if SMOKE_TEST else 30)   # the sweep's best config, more epochs
EMB_DIM = 768
SWEEP_12_EPOCHS = (0.9700, 0.9697)                        # same config at 12 epochs, retry v1 (first-5s, clip-max)
PRIOR_EPS = 1e-6                                          # as in the probe retry: prior clipped to [eps, 1-eps]

def find_dir(marker, env):
    if os.environ.get(env):
        return Path(os.environ[env])
    hits = glob.glob(f"/kaggle/input/**/{marker}", recursive=True)
    if not hits:
        raise FileNotFoundError(f"{marker} not found under /kaggle/input")
    return Path(hits[0]).parent

DATA = find_dir("birdclef-2025/sample_submission.csv", "M3_DATA_DIR")
MELS = find_dir("train_index.csv", "M3_MELS_DIR")
CACHE = find_dir("birdmae_embeddings.npy", "M3_EMBED_DIR")
print("Data:", DATA, "| M1 index:", MELS, "| embeddings:", CACHE)

## Split, labels and cached embeddings


In [ ]:
index = pd.read_csv(MELS / "train_index.csv")      # same table, in the same order, that M1 split
species = list(pd.read_csv(DATA / "sample_submission.csv", nrows=0).columns[1:])
sp2idx = {s: i for i, s in enumerate(species)}
y_idx = index["primary_label"].astype(str).map(sp2idx)
assert y_idx.notna().all(), "primary_label not in submission columns"
y_idx = y_idx.astype(int).values

def parse_secondary(s):
    try:
        labs = ast.literal_eval(s) if isinstance(s, str) else []
    except (ValueError, SyntaxError):
        labs = []
    return [sp2idx[l] for l in labs if l in sp2idx]

sec_idx = [parse_secondary(s) for s in index["secondary_labels"]]

# Copied from M1 train-v2 via the A3 notebook: same call, same array of clip numbers, same stratify, same seed.
idx_tr, idx_va = train_test_split(
    np.arange(len(index)), test_size=CFG["val_frac"], stratify=y_idx, random_state=CFG["seed"])
assert not set(idx_tr) & set(idx_va)
print(len(index), "clips,", len(species), "species | train", len(idx_tr), "| val", len(idx_va))

# The cache must belong to exactly this split and clip order
val_clips = pd.read_csv(CACHE / "val_clips.csv")
if not np.array_equal(val_clips["clip"].values, idx_va):
    raise RuntimeError("val_clips.csv does not match the recomputed split")
if "filename" in index.columns:
    if not (index["filename"].astype(str).values[idx_va] == val_clips["filename"].astype(str).values).all():
        raise RuntimeError("val_clips.csv filenames do not match train_index.csv")

E = np.load(CACHE / "birdmae_embeddings.npy")
windows = pd.read_csv(CACHE / "birdmae_windows.csv")
assert E.shape[1] == EMB_DIM and len(E) == len(windows), (E.shape, len(windows))
E = E.astype(np.float32)
print(f"Embeddings {E.shape} | {windows['clip'].nunique()} clips | loaded at {time.time() - T0:.0f}s", flush=True)

## Window weights, standardisation and validation sets (as in the sweep)


In [ ]:
clip_of, win_of = windows["clip"].values, windows["win"].values
in_train = np.isin(clip_of, idx_tr); in_val = np.isin(clip_of, idx_va)
assert not (in_train & in_val).any() and (in_train | in_val).all()

Y = np.zeros((len(windows), len(species)), np.float32)      # train: primary 1.0, secondary 0.5
for row, c in enumerate(clip_of):
    Y[row, sec_idx[c]] = CFG["secondary_weight"]
    Y[row, y_idx[c]] = 1.0
clip_windows = pd.Series(clip_of).map(pd.Series(clip_of).value_counts()).values
W_row = (1.0 / clip_windows).astype(np.float32)             # each clip counts once

mean, std = E[in_train].mean(axis=0), E[in_train].std(axis=0) + 1e-6
Z = (E - mean) / std

def macro_auc(y_true, y_pred):
    keep = y_true.sum(axis=0) > 0          # like the competition metric
    return roc_auc_score(y_true[:, keep], y_pred[:, keep], average="macro"), int(keep.sum())

# Validation rows, M1 protocol: first window, primary label only
rows_va = np.where(in_val & (win_of == 0))[0]
Y_va = np.zeros((len(rows_va), len(species)), np.float32)
Y_va[np.arange(len(rows_va)), y_idx[clip_of[rows_va]]] = 1.0

rows_all_va = np.where(in_val)[0]
va_groups = clip_of[rows_all_va]
order_clips = np.unique(va_groups)
Y_clip = np.zeros((len(order_clips), len(species)), np.float32)
Y_clip[np.arange(len(order_clips)), y_idx[order_clips]] = 1.0

X = torch.from_numpy(Z); Y_t = torch.from_numpy(Y); W_t = torch.from_numpy(W_row)
tr_rows = torch.from_numpy(np.where(in_train)[0])
print(f"Windows: train {in_train.sum()} | val {in_val.sum()} | val first-5s {len(rows_va)}", flush=True)

# Log prior of each class, as in the probe retry: same training rows, labels (secondary 0.5) and window weights as the loss
w_tr = W_row[in_train].astype(np.float64)
prior = (Y[in_train].astype(np.float64) * w_tr[:, None]).sum(axis=0) / w_tr.sum()
n_clipped = int(((prior < PRIOR_EPS) | (prior > 1 - PRIOR_EPS)).sum())
prior_clipped = np.clip(prior, PRIOR_EPS, 1 - PRIOR_EPS)
PRIOR_BIAS = torch.from_numpy(np.log(prior_clipped / (1 - prior_clipped)).astype(np.float32))
assert torch.isfinite(PRIOR_BIAS).all() and len(PRIOR_BIAS) == len(species)
print(f"Log-prior bias: mean {PRIOR_BIAS.mean():.4f} | min {PRIOR_BIAS.min():.4f} | max {PRIOR_BIAS.max():.4f} | "
      f"classes clipped {n_clipped}", flush=True)


## Head


In [ ]:
def make_head():
    return nn.Sequential(nn.Linear(EMB_DIM, HEAD["hidden"]), nn.ReLU(), nn.Dropout(HEAD["dropout"]), nn.Linear(HEAD["hidden"], len(species)))

@torch.no_grad()
def predict(model, rows):
    model.eval(); out = []
    for i in range(0, len(rows), 4096):
        out.append(torch.sigmoid(model(X[torch.from_numpy(rows[i:i + 4096])])).numpy())
    return np.concatenate(out)

def clip_max_auc(model):
    p = pd.DataFrame(predict(model, rows_all_va)).groupby(va_groups).max().loc[order_clips].values
    return macro_auc(Y_clip, p)[0]

def head_predict(head, emb):
    """Sigmoid probabilities from saved head arrays. Keep identical to the inference notebook."""
    x = (emb - head["mean"]) / head["std"]
    n_layers = int(head["n_layers"])
    for i in range(n_layers):
        x = x @ head[f"W{i}"].T + head[f"b{i}"]
        if i < n_layers - 1:
            x = np.maximum(x, 0.0)       # ReLU
    return 1.0 / (1.0 + np.exp(-x))


## Train


In [ ]:
def train_run(name, prior_bias):
    """One full training run. The seed is reset here, so both runs share initial weights, batches and dropout draws."""
    torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
    epochs = HEAD["epochs"]
    model = make_head()
    if prior_bias:
        with torch.no_grad():
            model[-1].bias.copy_(PRIOR_BIAS)     # no random draw, so the weights and batches match the default run's
    init_bias_mean = model[-1].bias.mean().item()
    print(f"[{name}] initial bias mean {init_bias_mean:.4f}", flush=True)
    steps = int(np.ceil(len(tr_rows) / CFG["batch_size"]))
    opt = torch.optim.AdamW(model.parameters(), lr=HEAD["max_lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=HEAD["max_lr"], total_steps=epochs * steps, pct_start=0.1)
    best = dict(first=-1.0); history = []; T_TRAIN = time.time()
    for ep in range(1, epochs + 1):
        model.train(); t0 = time.time(); total = 0.0
        perm = tr_rows[torch.randperm(len(tr_rows))]
        for s in range(steps):
            b = perm[s * CFG["batch_size"]:(s + 1) * CFG["batch_size"]]
            bce = nn.functional.binary_cross_entropy_with_logits(model(X[b]), Y_t[b], reduction="none")
            loss = (bce.mean(1) * W_t[b]).sum() / W_t[b].sum()
            opt.zero_grad(set_to_none=True)
            loss.backward(); opt.step(); sched.step()
            total += loss.item()
        t_train = time.time() - t0
        first, n_cls = macro_auc(Y_va, predict(model, rows_va))
        flag = ""
        if first > best["first"]:              # clip-max only when first-5s improves, as in the sweep
            best = dict(first=first, clip_max=clip_max_auc(model), best_epoch=ep, n_classes=n_cls,
                        state={k: v.clone() for k, v in model.state_dict().items()}); flag = "  <- best"
        sec = time.time() - t0
        history.append(dict(epoch=ep, loss=total / steps, first5s=first, seconds=sec))
        print(f"[{name}] ep {ep:2d}/{epochs} | loss {total / steps:.4f} | first-5s {first:.4f} | train {t_train:.1f}s | epoch {sec:.1f}s{flag}", flush=True)
    train_s = time.time() - T_TRAIN
    print(f"TIME [{name}] training {train_s:.0f}s ({train_s / epochs:.1f}s/epoch) | best epoch {best['best_epoch']} | first-5s {best['first']:.4f} | clip-max {best['clip_max']:.4f}", flush=True)
    return {**best, "init_bias_mean": init_bias_mean, "train_seconds": train_s, "history": history}

runs = {"default": train_run("default", prior_bias=False), "log_prior": train_run("log_prior", prior_bias=True)}
winning_bias = "log_prior" if runs["log_prior"]["first"] > runs["default"]["first"] else "default"   # default on a tie
best, history = runs[winning_bias], runs[winning_bias]["history"]
train_s = sum(r["train_seconds"] for r in runs.values())
model = make_head(); model.load_state_dict(best["state"]); model.eval()

print(f"\n{'bias':10s} | {'first-5s':>8s} | {'clip-max':>8s} | {'best ep':>7s} | {'init bias':>9s} | {'sec':>6s} | saved")
for name, r in runs.items():
    print(f"{name:10s} | {r['first']:8.4f} | {r['clip_max']:8.4f} | {r['best_epoch']:7d} | {r['init_bias_mean']:9.4f} | {r['train_seconds']:6.0f} | {'YES' if name == winning_bias else 'no'}")
print(f"\nWinning bias: {winning_bias}")
print(f"Same config at 12 epochs (retry v1): first-5s {SWEEP_12_EPOCHS[0]} | clip-max {SWEEP_12_EPOCHS[1]}")

## Save for inference and check the numpy path


In [ ]:
linears = [m for m in model if isinstance(m, nn.Linear)]
arrays = dict(mean=mean, std=std, n_layers=len(linears))
for i, m in enumerate(linears):
    arrays[f"W{i}"] = m.weight.detach().numpy(); arrays[f"b{i}"] = m.bias.detach().numpy()
# Not birdmae_head.npz: the embed notebook's linear probe has that name, and inference attaches both kernels.
np.savez(OUT / "birdmae_mlp_head.npz", **arrays)
json.dump(species, open(OUT / "species.json", "w"))

# The numpy path used at inference must reproduce the trained torch head
head = dict(np.load(OUT / "birdmae_mlp_head.npz"))
assert np.allclose(head_predict(head, E[rows_va]), predict(model, rows_va), atol=1e-4), "numpy head differs from torch head (val embeddings)"
rng = np.random.default_rng(CFG["seed"])
emb_rand = (mean + std * rng.standard_normal((256, EMB_DIM))).astype(np.float32)
with torch.no_grad():
    p_torch = torch.sigmoid(model(torch.from_numpy(((emb_rand - mean) / std).astype(np.float32)))).numpy()
p_np = head_predict(head, emb_rand)
max_diff = float(np.abs(p_np - p_torch).max())
assert max_diff < 1e-4, f"numpy head differs from torch head (random input): {max_diff}"
print(f"numpy vs torch head on 256 random inputs: max abs diff {max_diff:.2e}")

# Inference checks the Bird-MAE files against these hashes, so they are carried over from the embed run
embed_cfg = json.load(open(CACHE / "head_cfg.json"))
summary = dict(best_head="mlp", val_auc_first5s=float(best["first"]), val_auc_clip_max=float(best["clip_max"]), val_classes=best["n_classes"],
               best_epoch=best["best_epoch"], head=HEAD, cfg=CFG, train_seconds=train_s, history=[{k: float(v) for k, v in h.items()} for h in history],
               hf_repo=embed_cfg["hf_repo"], hf_revision=embed_cfg["hf_revision"],
               birdmae_files_sha256=embed_cfg["birdmae_files_sha256"], transformers_pin=embed_cfg["transformers_pin"],
               winning_bias=winning_bias, prior_eps=PRIOR_EPS, prior_classes_clipped=n_clipped,
               runs={name: dict(val_auc_first5s=float(r["first"]), val_auc_clip_max=float(r["clip_max"]), best_epoch=r["best_epoch"], val_classes=r["n_classes"],
                                init_bias_mean=r["init_bias_mean"], train_seconds=r["train_seconds"],
                                history=[{k: float(v) for k, v in h.items()} for h in r["history"]]) for name, r in runs.items()})
json.dump(summary, open(OUT / "head_cfg.json", "w"), indent=1)
print(f"Saved birdmae_mlp_head.npz, species.json, head_cfg.json to {OUT} | notebook total {time.time() - T0:.0f}s")
